In [14]:
import os
from docling.document_converter import DocumentConverter
import os
from dotenv import load_dotenv

In [2]:
source = "/Users/karim/Desktop/laws for chatbot/Civil-Code.pdf"
converter = DocumentConverter()
doc= converter.convert(source)

In [3]:
md_output = doc.document.export_to_markdown()

In [4]:
print(md_output)

## 89/2012 Sb. ACT

## of 3 February 2012

## the Civil Code

the Parliament has adopted the following Act of the Czech Republic:

## BOOK ONE GENERAL PROVISIONS

## TITLE I

## SCOPE OF REGULATION AND ITS BASIC PRINCIPLES

## Chapter 1

## Private law

Section 1 [Recodification]

(1) The provisions of the legal order governing the mutual rights and duties of persons together constitute private law.

The application of private law is independent of the application of public law. (2)  Unless  expressly  prohibited  by  a  statute,  persons  can  stipulate  rights  and  duties  by  way  of  exclusion  from  a statute; stipulations  contrary  to  good  morals,  public  order  or  the  law  concerning  the  status  of  persons,  including  the  right  to protection of personality rights, are prohibited. Section 2 [Recodification] (1) Each provision of private law may be interpreted only in accordance with the Charter of Fundamental Rights and Freedoms and the constitutional order in genera

In [ ]:
import re
from collections import OrderedDict
def preprocess_markdown(text):
    new_lines = []
    for line in text.split("\n"):
        parts = re.split(r"(Section\s+\d+)", line, flags=re.IGNORECASE)
        if len(parts) <= 2:
            new_lines.append(line)
        else:
            current = parts[0]
            for i in range(1, len(parts)):
                if re.match(r"Section\s+\d+", parts[i], re.IGNORECASE):
                    if current.strip():
                        new_lines.append(current)
                    current = parts[i]
                else:
                    current += parts[i]
            if current.strip():
                new_lines.append(current)
    return "\n".join(new_lines)

def parse_legal_text_final(text, law_name="Business Corporations Act (90/2012)"):
    text = preprocess_markdown(text)

    section_pattern = re.compile(
        r"^(?:#+\s*)?(?:-\s*)?Section\s+(\d+)\s*(?:\[.*?\])?\s*(.*)",
        re.IGNORECASE
    )
    chapter_pattern = re.compile(r"^(?:#+\s*)?Chapter\s+(\d+)", re.IGNORECASE)
    division_pattern = re.compile(r"^(?:#+\s*)?Division\s+(\d+)", re.IGNORECASE)
    subdivision_pattern = re.compile(r"^(?:#+\s*)?Subdivision\s+(\d+)", re.IGNORECASE)
    skip_patterns = [
        re.compile(r"^(?:#+\s*)?TITLE\s+", re.IGNORECASE),
        re.compile(r"^(?:#+\s*)?PART\s+", re.IGNORECASE),
        re.compile(r"^#{1,6}\s*$"),
        re.compile(r"^(?:#+\s*)?BUSINESS\s+CORPORATIONS", re.IGNORECASE),
        re.compile(r"^(?:#+\s*)?COOPERATIVE\s*$", re.IGNORECASE),
    ]

    current_chapter_id = None
    current_chapter_name = None
    current_division_id = None
    current_division_name = None
    expecting_chapter_name = False
    expecting_division_name = False
    current_section_id = None
    current_text_lines = []
    all_chunks = []

    def clean_md(line):
        return re.sub(r"^#+\s*", "", line).strip()

    def should_skip(line):
        for p in skip_patterns:
            if p.match(line):
                return True
        return False

    def save_chunk():
        if current_section_id is None:
            return
        text_joined = " ".join(current_text_lines).strip()
        if not text_joined:
            return
        all_chunks.append({
            "law_name": law_name,
            "chapter_id": current_chapter_id or "Unknown",
            "chapter_name": current_chapter_name or "Unknown",
            "division_id": current_division_id or "None",
            "division_name": current_division_name or "None",
            "section_id": current_section_id,
            "text": text_joined,
        })

    for line in text.split("\n"):
        stripped = line.strip()
        if not stripped:
            continue

        section_match = section_pattern.match(stripped)
        if section_match:
            save_chunk()
            current_section_id = f"Section {section_match.group(1)}"
            current_text_lines = []
            expecting_chapter_name = False
            expecting_division_name = False
            remaining = section_match.group(2).strip()
            if remaining:
                current_text_lines.append(remaining)
            continue

        chapter_match = chapter_pattern.match(stripped)
        if chapter_match:
            save_chunk()
            current_section_id = None
            current_text_lines = []
            current_chapter_id = f"Chapter {chapter_match.group(1)}"
            current_chapter_name = None
            current_division_id = None
            current_division_name = None
            expecting_chapter_name = True
            expecting_division_name = False
            after = re.sub(r"^(?:#+\s*)?Chapter\s+\d+\s*", "", stripped, flags=re.IGNORECASE).strip()
            if after and not division_pattern.match(after) and not section_pattern.match(after):
                current_chapter_name = clean_md(after)
                expecting_chapter_name = False
            continue

        division_match = division_pattern.match(stripped)
        if division_match:
            current_division_id = f"Division {division_match.group(1)}"
            current_division_name = None
            expecting_division_name = True
            expecting_chapter_name = False
            after = re.sub(r"^(?:#+\s*)?Division\s+\d+\s*", "", stripped, flags=re.IGNORECASE).strip()
            if after:
                current_division_name = clean_md(after)
                expecting_division_name = False
            continue

        if subdivision_pattern.match(stripped):
            continue

        if should_skip(stripped):
            continue

        clean = clean_md(stripped)
        if not clean:
            continue

        if expecting_chapter_name:
            current_chapter_name = clean
            expecting_chapter_name = False
            continue

        if expecting_division_name:
            current_division_name = clean
            expecting_division_name = False
            continue

        if current_section_id is not None:
            current_text_lines.append(stripped)

    save_chunk()

    for chunk in all_chunks:
        t = chunk["text"]
        t = re.sub(r"\s{2,}", " ", t)
        t = t.replace("compan ies", "companies")
        t = t.replace("partnership s", "partnerships")
        t = t.replace("jointstock", "joint-stock")
        t = t.replace(" -liability", "-liability")
        chunk["text"] = t.strip()
        chunk["chapter_name"] = re.sub(r"^#+\s*", "", chunk["chapter_name"]).strip() or "Unknown"

    merged = OrderedDict()
    for c in all_chunks:
        sid = c["section_id"]
        if sid not in merged:
            merged[sid] = c.copy()
        else:
            existing_text = merged[sid]["text"]
            new_text = c["text"]
            if new_text not in existing_text:
                merged[sid]["text"] = existing_text + " " + new_text

    chunks = sorted(merged.values(), key=lambda c: int(c["section_id"].split()[-1]))
    return chunks


chunks = parse_legal_text_final(md_output)
print(f"Total sections parsed: {len(chunks)}")

all_nums = sorted([int(c["section_id"].split()[-1]) for c in chunks])
missing = [i for i in range(all_nums[0], all_nums[-1] + 1) if i not in all_nums]
null_ch = [c for c in chunks if c["chapter_name"] in (None, "Unknown", "")]
empty_text = [c for c in chunks if len(c["text"].strip()) < 10]

print(f"Section range: {all_nums[0]} — {all_nums[-1]}")
print(f"Missing sections: {len(missing)}")
print(f"Unknown chapter names: {len(null_ch)}")
print(f"Empty/short chunks: {len(empty_text)}")

if missing:
    print(f"\nStill missing: {missing}")

if empty_text:
    print(f"\nShort chunks:")
    for c in empty_text[:5]:
        print(f"  {c['section_id']}: '{c['text']}'")

print(f"\n--- Spot checks ---")
for n in [1, 95, 96, 97, 98, 127, 220, 221, 343, 552, 553, 554, 704, 786]:
    found = [c for c in chunks if c["section_id"] == f"Section {n}"]
    if found:
        c = found[0]
        print(f"Section {n}: {c['chapter_id']} | {c['text'][:70]}...")
    else:
        print(f"Section {n}: MISSING")

Total sections parsed: 3071
Section range: 1 — 3081
Missing sections: 10
Unknown chapter names: 0
Empty/short chunks: 1

Still missing: [523, 544, 972, 1373, 1541, 1619, 1894, 2511, 2780, 3044]

Short chunks:
  Section 374: '(1)'

--- Spot checks ---
Section 1: Chapter 1 | (1) The provisions of the legal order governing the mutual rights and ...
Section 95: Chapter 2 | A minor without full legal capacity may, in usual matters, also give h...
Section 96: Chapter 2 | - (1) The consent to an interference with the integrity of an individu...
Section 97: Chapter 2 | - (1) The consent granted may be withdrawn in any form, even where the...
Section 98: Chapter 2 | - (1) If a person cannot give consent due to the inability, even tempo...
Section 127: Chapter 3 | Acts in the name of a legal person may even be performed prior to its ...
Section 220: Chapter 3 | - (1) If the articles of association provide various kinds of membersh...
Section 221: Chapter 3 | The articles of association must be k

In [6]:
import chromadb
from sentence_transformers import SentenceTransformer

In [7]:
client = chromadb.PersistentClient(path="./data/civil_code_db")
collection = client.get_or_create_collection(
    name="czech_business_law",
    metadata={"hnsw:space": "cosine"},)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
text = [c["text"] for c in chunks]
embedings = model.encode(text) 
print(embedings.shape)

(3071, 384)


In [ ]:
all_ids = []
all_documents = []
all_metadatas = []
all_embeddings = embedings.tolist()   

for i, chunk in enumerate(chunks):

    all_ids.append(f"sec_{i}")

    all_documents.append(chunk["text"])


    all_metadatas.append({
        "law_name":      chunk.get("law_name", "Unknown") or "Unknown",
        "chapter_id":    chunk.get("chapter_id", "Unknown") or "Unknown",
        "chapter_name":  chunk.get("chapter_name", "Unknown") or "Unknown",
        "division_id":   chunk.get("division_id", "Unknown") or "Unknown",
        "division_name": chunk.get("division_name", "Unknown") or "Unknown",
        "section_id":    chunk.get("section_id", "Unknown") or "Unknown",
    })


batch_size = 50

for start in range(0, len(all_ids), batch_size):
    end = start + batch_size

    collection.add(
        ids=all_ids[start:end],
        documents=all_documents[start:end],
        metadatas=all_metadatas[start:end],
        embeddings=all_embeddings[start:end],
    )

    done = min(end, len(all_ids))
    print(f"Added {done}/{len(all_ids)}")

print(f"\nDone! Collection now has {collection.count()} items.")

Added 50/3071
Added 100/3071
Added 150/3071
Added 200/3071
Added 250/3071
Added 300/3071
Added 350/3071
Added 400/3071
Added 450/3071
Added 500/3071
Added 550/3071
Added 600/3071
Added 650/3071
Added 700/3071
Added 750/3071
Added 800/3071
Added 850/3071
Added 900/3071
Added 950/3071
Added 1000/3071
Added 1050/3071
Added 1100/3071
Added 1150/3071
Added 1200/3071
Added 1250/3071
Added 1300/3071
Added 1350/3071
Added 1400/3071
Added 1450/3071
Added 1500/3071
Added 1550/3071
Added 1600/3071
Added 1650/3071
Added 1700/3071
Added 1750/3071
Added 1800/3071
Added 1850/3071
Added 1900/3071
Added 1950/3071
Added 2000/3071
Added 2050/3071
Added 2100/3071
Added 2150/3071
Added 2200/3071
Added 2250/3071
Added 2300/3071
Added 2350/3071
Added 2400/3071
Added 2450/3071
Added 2500/3071
Added 2550/3071
Added 2600/3071
Added 2650/3071
Added 2700/3071
Added 2750/3071
Added 2800/3071
Added 2850/3071
Added 2900/3071
Added 2950/3071
Added 3000/3071
Added 3050/3071
Added 3071/3071

Done! Collection now has 30

In [24]:
def retrieve(query, top_k=5):
    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
    )

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "section_id": results["metadatas"][0][i]["section_id"],
            "chapter_id": results["metadatas"][0][i]["chapter_id"],
            "chapter_name": results["metadatas"][0][i]["chapter_name"],
            "division_name": results["metadatas"][0][i].get("division_name", ""),
            "text": results["documents"][0][i],
            "distance": round(results["distances"][0][i], 4),
        })
    return retrieved

In [34]:
from google import genai
load_dotenv()
api=os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=api)


def generate(query, context_chunks):
    context = ""
    for c in context_chunks:
        label = f"{c['chapter_id']}, {c['section_id']}"
        if c.get("division_name") and c["division_name"] != "Unknown":
            label += f", {c['division_name']}"
        context += f"\n[{label}]\n{c['text']}\n"

        prompt = f"""You are a legal assistant specializing in Czech business law (Act 90/2012 - Business Corporations Act).

            STRICT RULES:
            1. Answer ONLY based on the legal excerpts provided below
            2. Always cite the specific Section number in your answer
            3. If the answer is NOT found in the excerpts, say: "This question is not covered in the provided legal sections."
            4. Do NOT use any outside legal knowledge
            5. Keep your answer practical and concise

            LEGAL EXCERPTS:
            {context}

            USER QUESTION: {query}"""
        
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text

In [35]:
def ask(question, top_k=5):
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    results = retrieve(question, top_k)

    print(f"\nRetrieved {len(results)} sources:")
    for r in results:
        print(f"  [{r['distance']:.3f}] {r['chapter_id']} / {r['section_id']}")

    answer = generate(question, results)

    print(f"\nAnswer:\n{answer}")
    print(f"{'='*60}")
    return answer

In [36]:
ask("I bought a warehouse space for my company in Prague, and three months later, I discovered the roof was leaking, even though the seller swore everything was fine. What's the legal deadline for filing a claim, and can I cancel the deal?")


Question: I bought a warehouse space for my company in Prague, and three months later, I discovered the roof was leaking, even though the seller swore everything was fine. What's the legal deadline for filing a claim, and can I cancel the deal?

Retrieved 5 sources:
  [0.550] Chapter 7 / Section 2569
  [0.573] Chapter 7 / Section 1923
  [0.586] Chapter 3 / Section 2939
  [0.588] Chapter 1 / Section 2129
  [0.594] Chapter 4 / Section 2407

Answer:
For a latent defect associated with a structure firmly attached to the tract of land, such as a leaking roof in a warehouse, the buyer must notify the seller of the defect within **five years from the date of acquisition**. (Section 2129(2))

However, the seller is not entitled to assert a defense of late notification if the defect is due to a fact of which the seller was or must have been aware at the time the thing was delivered. (Section 2129(2))

This question is not covered in the provided legal sections.


'For a latent defect associated with a structure firmly attached to the tract of land, such as a leaking roof in a warehouse, the buyer must notify the seller of the defect within **five years from the date of acquisition**. (Section 2129(2))\n\nHowever, the seller is not entitled to assert a defense of late notification if the defect is due to a fact of which the seller was or must have been aware at the time the thing was delivered. (Section 2129(2))\n\nThis question is not covered in the provided legal sections.'

In [ ]:
from docling.document_converter import DocumentConverter

source = "/Users/karim/Desktop/laws for chatbot/trade.pdf"
converter = DocumentConverter()
doc_trade = converter.convert(source)
md_trade = doc_trade.document.export_to_markdown()

lines = md_trade.split("\n")
print(f"Total lines: {len(lines)}")
print()
for i, line in enumerate(lines[:100]):
    if line.strip():
        print(f"{i}: {repr(line.strip())}")

Total lines: 1492

0: 'The Federal Assembly of the Czech and Slovak Federative Republic has passed the following Act:'
2: '## GENERAL PROVISIONS'
4: '## TITLE I'
6: '## SUBJECT OF REGULATION'
8: 'Section 1'
10: "This Act lays down conditions for carrying on a licensed trade (hereinafter referred to as ' trade ' ) and inspections of compliance with those conditions."
12: '## Trades'
14: 'Section 2'
16: "A  trade  shall  mean  a  systematic  activity  carried  out  independently  under  the  conditions  laid down in this Act, under a person's own name and liability, with a view to making a profit."
18: '## Section 3'
20: '(1) The following shall not constitute a trade:'
22: '- a) the performance of an activity statutorily reserved for the State or for a designated legal person,'
23: '- b) the  use  of  the  results  of  intellectual  creativity  protected  by  specific  laws,  their  originators  or authors 2) ,'
24: '- c) the  collective  administration  of  copyright  and  rights  rela

In [ ]:
import re
section_lines = [(i, repr(line.strip())[:120]) for i, line in enumerate(lines) if re.search(r"Section\s+\d", line)]
print(f"Total section-like lines: {len(section_lines)}")
print("\nFirst 15:")
for idx, txt in section_lines[:15]:
    print(f"  Line {idx}: {txt}")
print("\nLast 10:")
for idx, txt in section_lines[-10:]:
    print(f"  Line {idx}: {txt}")

Total section-like lines: 246

First 15:
  Line 8: 'Section 1'
  Line 14: 'Section 2'
  Line 18: '## Section 3'
  Line 34: '2c) Section 21(2) of Act No 20/1987 Coll., on the care of monuments by the State.'
  Line 72: '9) Section  11  and  Section  13(1)  of  Act  No  2/1991  Coll.,  on  collective  bargaining,  as  amended  by  Act  No
  Line 78: '10a) Section 14(1)(a) of Act No 360/1992 Coll., on the profession of authorised architects and the profession of author
  Line 80: '10b) Section 144(4) of Act No 183/2006 Coll., on land-use planning and building rules (the Building Act).'
  Line 134: '18) Section 60(3) of Act No 266/1994 Coll., on railways.'
  Line 173: '23h) Section 27 of Act No 250/2000 Coll., on budgetary rules of territorial budgets.'
  Line 175: '23i) Section 4(2)(b) and Sections 48 to 50 of Act No 359/1999 Coll., on child protection.'
  Line 206: '56) Section  81  of  Act  No  326/2004  Coll.,  on  phytosanitary  care  and  amending  certain  related  acts,  as ame
  L

In [ ]:
def preprocess_trade_act(text):
    new_lines = []
    for line in text.split("\n"):
        stripped = line.strip()

        if re.match(r"^\d+[a-z]?\)\s", stripped):
            continue

        parts = re.split(r"(Section\s+\d+)", line, flags=re.IGNORECASE)
        if len(parts) <= 2:
            new_lines.append(line)
        else:
            current = parts[0]
            for i in range(1, len(parts)):
                if re.match(r"Section\s+\d+", parts[i], re.IGNORECASE):
                    if current.strip():
                        new_lines.append(current)
                    current = parts[i]
                else:
                    current += parts[i]
            if current.strip():
                new_lines.append(current)

    return "\n".join(new_lines)


def parse_trade_act(text, law_name="Trade Licensing Act (455/1991)"):
    text = preprocess_trade_act(text)

    section_pattern = re.compile(
        r"^(?:#+\s*)?(?:-\s*)?Section\s+(\d+)\s*(?:\[.*?\])?\s*(.*)",
        re.IGNORECASE
    )
    title_pattern = re.compile(r"^(?:#+\s*)?TITLE\s+([IVXLC]+)", re.IGNORECASE)
    part_pattern = re.compile(r"^(?:#+\s*)?PART\s+(\w+)", re.IGNORECASE)
    division_pattern = re.compile(r"^(?:#+\s*)?Division\s+(\d+)", re.IGNORECASE)
    skip_patterns = [
        re.compile(r"^#{1,6}\s*$"),
        re.compile(r"^-\s*\d+\s*-\s*$"), 
    ]

    current_part_id = None
    current_part_name = None
    current_title_id = None
    current_title_name = None
    current_division_id = None
    current_division_name = None
    expecting_part_name = False
    expecting_title_name = False
    expecting_division_name = False
    current_section_id = None
    current_text_lines = []
    all_chunks = []

    def clean_md(line):
        return re.sub(r"^#+\s*", "", line).strip()

    def should_skip(line):
        for p in skip_patterns:
            if p.match(line):
                return True
        return False

    def save_chunk():
        if current_section_id is None:
            return
        text_joined = " ".join(current_text_lines).strip()

        
        if not text_joined or text_joined.lower() == "(deleted)" or text_joined.lower() == "deleted":
            return

        all_chunks.append({
            "law_name": law_name,
            "part_id": current_part_id or "Unknown",
            "part_name": current_part_name or "Unknown",
            "title_id": current_title_id or "Unknown",
            "title_name": current_title_name or "Unknown",
            "division_id": current_division_id or "None",
            "division_name": current_division_name or "None",
            "section_id": current_section_id,
            "text": text_joined,
        })

    for line in text.split("\n"):
        stripped = line.strip()
        if not stripped:
            continue

        if re.match(r"^\d+[a-z]?\)\s", stripped):
            continue

        section_match = section_pattern.match(stripped)
        if section_match:
            save_chunk()
            current_section_id = f"Section {section_match.group(1)}"
            current_text_lines = []
            expecting_part_name = False
            expecting_title_name = False
            expecting_division_name = False
            remaining = section_match.group(2).strip()
            if remaining:
                current_text_lines.append(remaining)
            continue

        part_match = part_pattern.match(stripped)
        if part_match:
            save_chunk()
            current_section_id = None
            current_text_lines = []
            current_part_id = f"Part {part_match.group(1)}"
            current_part_name = None
            current_title_id = None
            current_title_name = None
            current_division_id = None
            current_division_name = None
            expecting_part_name = True
            expecting_title_name = False
            expecting_division_name = False
            continue

        title_match = title_pattern.match(stripped)
        if title_match:
            current_title_id = f"Title {title_match.group(1)}"
            current_title_name = None
            current_division_id = None
            current_division_name = None
            expecting_title_name = True
            expecting_part_name = False
            expecting_division_name = False
            continue

        division_match = division_pattern.match(stripped)
        if division_match:
            current_division_id = f"Division {division_match.group(1)}"
            current_division_name = None
            expecting_division_name = True
            expecting_part_name = False
            expecting_title_name = False
            continue

        if should_skip(stripped):
            continue

        clean = clean_md(stripped)
        if not clean:
            continue

        if expecting_part_name:
            current_part_name = clean
            expecting_part_name = False
            continue

        if expecting_title_name:
            current_title_name = clean
            expecting_title_name = False
            continue

        if expecting_division_name:
            current_division_name = clean
            expecting_division_name = False
            continue

        if current_section_id is not None:
            current_text_lines.append(stripped)

    save_chunk()

    for chunk in all_chunks:
        t = chunk["text"]
        t = re.sub(r"\s{2,}", " ", t)
        chunk["text"] = t.strip()

    merged = OrderedDict()
    for c in all_chunks:
        sid = c["section_id"]
        if sid not in merged:
            merged[sid] = c.copy()
        else:
            existing = merged[sid]["text"]
            new_text = c["text"]
            if new_text not in existing:
                merged[sid]["text"] = existing + " " + new_text

    chunks = sorted(merged.values(), key=lambda c: int(c["section_id"].split()[-1]))
    return chunks


chunks_trade = parse_trade_act(md_trade)
print(f"Total sections parsed: {len(chunks_trade)}")


all_nums = sorted([int(c["section_id"].split()[-1]) for c in chunks_trade])
missing = [i for i in range(all_nums[0], all_nums[-1] + 1) if i not in all_nums]

print(f"Section range: {all_nums[0]} — {all_nums[-1]}")
print(f"Missing sections: {len(missing)}")

if missing:
    print(f"Missing: {missing}")

deleted_in_pdf = [4, 6, 7, 12, 15, 16, 29, 30, 31, 32, 33, 35, 36, 37, 38, 39, 40, 41, 51, 59, 65, 66, 69]
print(f"\nKnown deleted sections: {len(deleted_in_pdf)}")
actual_missing = [m for m in missing if m not in deleted_in_pdf]
print(f"Unexpectedly missing (not deleted): {actual_missing if actual_missing else 'NONE'}")

print(f"\n--- Spot checks ---")
for n in [1, 2, 3, 5, 6, 8, 17, 31, 45, 60, 73, 81]:
    found = [c for c in chunks_trade if c["section_id"] == f"Section {n}"]
    if found:
        c = found[0]
        print(f"Section {n}: {c['part_id']} / {c['title_id']} | {c['text'][:70]}...")
    else:
        print(f"Section {n}: MISSING (likely deleted)")


Total sections parsed: 75
Section range: 1 — 81
Missing sections: 6
Missing: [15, 29, 39, 40, 43, 66]

Known deleted sections: 23
Unexpectedly missing (not deleted): [43]

--- Spot checks ---
Section 1: Unknown / Title I | This Act lays down conditions for carrying on a licensed trade (herein...
Section 2: Unknown / Title I | A trade shall mean a systematic activity carried out independently und...
Section 3: Unknown / Title I | (1) The following shall not constitute a trade: - a) the performance o...
Section 5: Unknown / Title II | ## Entities eligible to carry on a trade (1) A natural or legal person...
Section 6: Unknown / Title II | ## General conditions for carrying on a trade (1) Unless otherwise pro...
Section 8: Unknown / Title II | ## Impediments to carrying on a trade (1) A natural or legal person wh...
Section 17: Unknown / Title II | (1) For the purposes of this Act, an establishment shall mean the spac...
Section 31: Unknown / Title II | (11)). (9) An entrepreneur may also